# Four-Parameter Cat-Qubit Optimizer

This notebook builds a rudimentary `dynamiqs`-based optimization model for the four real control knobs
`Re(g_2)`, `Im(g_2)`, `Re(\epsilon_d)`, and `Im(\epsilon_d)`.

The workflow is intentionally simple:
1. simulate logical `X` and `Z` decay under a chosen set of controls
2. fit proxy lifetimes `T_X` and `T_Z`
3. define a reward that prefers larger lifetimes and a target bias `T_Z / T_X`
4. use CMA-ES to search over the four real controls

The final cells produce several useful graphs:
- best reward per epoch
- optimizer mean trajectory for all four controls
- estimated `T_X`, `T_Z`, and bias over time
- scatter plots of sampled `g_2` and `\epsilon_d` controls colored by reward
- logical decay curves for the best candidate

In [ ]:
# If needed, install dependencies from the repo root with:
# !pip install -r ../requirements.txt

import dynamiqs as dq
import jax.numpy as jnp
from matplotlib import pyplot as plt
from scipy.optimize import least_squares
from cmaes import SepCMA

In [ ]:
def model(p, t):
    A, tau, C = p
    return A * jnp.exp(-t / tau) + C


def residuals(p, x, y):
    return model(p, x) - y


def robust_exp_fit(x, y):
    A0 = y.max() - y.min()
    C0 = y.min()
    tau0 = x.max() - x.min()
    p0 = [A0, tau0, C0]

    res = least_squares(
        residuals,
        p0,
        args=(x, y),
        bounds=([0, 0, -jnp.inf], [jnp.inf, jnp.inf, jnp.inf]),
        loss="soft_l1",
        f_scale=0.1,
    )

    return {
        "popt": res.x,
        "y_fit": model(res.x, x),
    }


def unpack_cat_controls(p):
    p = [float(x) for x in p]
    g2 = p[0] + 1j * p[1]
    eps_d = p[2] + 1j * p[3]
    return g2, eps_d


def reference_cat_amplitude(g2, eps_d, kappa_a=1.0, kappa_b=10.0):
    kappa_2 = max(4 * abs(g2) ** 2 / kappa_b, 1e-4)
    eps_2 = 2 * g2 * eps_d / kappa_b
    alpha_sq = (2 * eps_2 / kappa_2) - (kappa_a / (2 * kappa_2))
    alpha_estimate = complex(jnp.sqrt(jnp.asarray(alpha_sq, dtype=jnp.complex64)))

    if abs(alpha_estimate) < 0.5:
        alpha_estimate = 0.5 + 0.0j

    return alpha_estimate, eps_2, kappa_2


def measure_lifetime_with_controls(
    initial_state,
    tfinal,
    g2,
    eps_d,
    *,
    kappa_b=10.0,
    kappa_a=1.0,
    na=15,
    nb=5,
    nsave=64,
):
    a_storage = dq.destroy(na)
    a = dq.tensor(a_storage, dq.eye(nb))
    b = dq.tensor(dq.eye(na), dq.destroy(nb))

    alpha_estimate, _, kappa_2 = reference_cat_amplitude(g2, eps_d, kappa_a=kappa_a, kappa_b=kappa_b)

    H = (
        jnp.conj(g2) * a @ a @ b.dag()
        + g2 * a.dag() @ a.dag() @ b
        - eps_d * b.dag()
        - jnp.conj(eps_d) * b
    )

    loss_b = jnp.sqrt(kappa_b) * b
    loss_a = jnp.sqrt(kappa_a) * a
    tsave = jnp.linspace(0, tfinal, nsave)

    g_state = dq.coherent(na, alpha_estimate)
    e_state = dq.coherent(na, -alpha_estimate)

    basis = {
        "+z": g_state,
        "-z": e_state,
        "+x": (g_state + e_state) / jnp.sqrt(2),
        "-x": (g_state - e_state) / jnp.sqrt(2),
    }

    sx_storage = (1j * jnp.pi * a_storage.dag() @ a_storage).expm()
    sz_storage = basis["+z"] @ basis["+z"].dag() - basis["-z"] @ basis["-z"].dag()

    exp_ops = [
        dq.tensor(sx_storage, dq.eye(nb)),
        dq.tensor(sz_storage, dq.eye(nb)),
    ]

    psi0 = dq.tensor(basis[initial_state], dq.fock(nb, 0))

    res = dq.mesolve(
        H,
        [loss_b, loss_a],
        psi0,
        tsave,
        options=dq.Options(progress_meter=False),
        exp_ops=exp_ops,
    )

    return res, alpha_estimate, kappa_2


def evaluate_cat_controls(
    p,
    *,
    target_bias=25.0,
    kappa_b=10.0,
    kappa_a=1.0,
    z_tfinal=80.0,
    x_tfinal=2.0,
    return_results=False,
):
    g2, eps_d = unpack_cat_controls(p)

    try:
        res_z, alpha_estimate, kappa_2 = measure_lifetime_with_controls(
            "+z",
            z_tfinal,
            g2,
            eps_d,
            kappa_b=kappa_b,
            kappa_a=kappa_a,
        )
        res_x, _, _ = measure_lifetime_with_controls(
            "+x",
            x_tfinal,
            g2,
            eps_d,
            kappa_b=kappa_b,
            kappa_a=kappa_a,
        )

        fit_z = robust_exp_fit(res_z.tsave, res_z.expects[1, :].real)
        fit_x = robust_exp_fit(res_x.tsave, res_x.expects[0, :].real)

        Tz = max(float(fit_z["popt"][1]), 1e-4)
        Tx = max(float(fit_x["popt"][1]), 1e-4)
        bias = Tz / Tx

        lifetime_score = 0.5 * (jnp.log10(Tx + 1e-6) + jnp.log10(Tz + 1e-6))
        bias_penalty = jnp.log10((bias + 1e-6) / target_bias) ** 2
        reward = float(lifetime_score - 0.30 * bias_penalty)

        summary = {
            "loss": -reward,
            "reward": reward,
            "Tx": Tx,
            "Tz": Tz,
            "bias": bias,
            "alpha_abs": abs(alpha_estimate),
            "kappa_2": float(kappa_2),
            "g2": g2,
            "eps_d": eps_d,
        }

        if return_results:
            summary["result_z"] = res_z
            summary["result_x"] = res_x
            summary["fit_z"] = fit_z
            summary["fit_x"] = fit_x

        return summary

    except Exception as exc:
        summary = {
            "loss": 5.0,
            "reward": -5.0,
            "Tx": jnp.nan,
            "Tz": jnp.nan,
            "bias": jnp.nan,
            "alpha_abs": jnp.nan,
            "kappa_2": jnp.nan,
            "g2": g2,
            "eps_d": eps_d,
            "error": str(exc),
        }
        if return_results:
            summary["result_z"] = None
            summary["result_x"] = None
            summary["fit_z"] = None
            summary["fit_x"] = None
        return summary

In [ ]:
TARGET_BIAS = 25.0
KAPPA_A = 1.0
KAPPA_B = 10.0
BATCH_SIZE = 4
N_EPOCHS = 8

mean0 = jnp.array([1.0, 0.0, 4.0, 0.0])
sigma0 = 0.30
bounds = jnp.array([
    [0.50, 1.50],
    [-1.00, 1.00],
    [2.00, 6.00],
    [-2.00, 2.00],
])

optimizer = SepCMA(
    mean=mean0,
    sigma=sigma0,
    bounds=bounds,
    population_size=BATCH_SIZE,
    seed=7,
)

mean_history = []
epoch_best_reward_history = []
epoch_best_tx_history = []
epoch_best_tz_history = []
epoch_best_bias_history = []
sample_history = []
sample_reward_history = []

best_summary = None

for epoch in range(N_EPOCHS):
    xs = [optimizer.ask() for _ in range(optimizer.population_size)]
    metrics = [
        evaluate_cat_controls(
            x,
            target_bias=TARGET_BIAS,
            kappa_a=KAPPA_A,
            kappa_b=KAPPA_B,
        )
        for x in xs
    ]
    losses = [float(m["loss"]) for m in metrics]

    optimizer.tell(list(zip(xs, losses)))

    epoch_best_idx = int(jnp.argmin(jnp.asarray(losses)))
    epoch_best_metric = metrics[epoch_best_idx]
    epoch_best_params = jnp.asarray(xs[epoch_best_idx])

    if best_summary is None or epoch_best_metric["reward"] > best_summary["reward"]:
        best_summary = {
            "epoch": epoch,
            "params": epoch_best_params,
            **epoch_best_metric,
        }

    mean_history.append(jnp.asarray(optimizer.mean))
    epoch_best_reward_history.append(epoch_best_metric["reward"])
    epoch_best_tx_history.append(epoch_best_metric["Tx"])
    epoch_best_tz_history.append(epoch_best_metric["Tz"])
    epoch_best_bias_history.append(epoch_best_metric["bias"])

    sample_history.extend([jnp.asarray(x) for x in xs])
    sample_reward_history.extend([m["reward"] for m in metrics])

    print(
        "Epoch {:02d} | reward={:.3f} | Tx={:.3f} us | Tz={:.3f} us | bias={:.2f}".format(
            epoch,
            epoch_best_metric["reward"],
            epoch_best_metric["Tx"],
            epoch_best_metric["Tz"],
            epoch_best_metric["bias"],
        )
    )

best_params = best_summary["params"]
print()
print("Best parameters found:")
print("  Re(g_2)       = {:.4f}".format(float(best_params[0])))
print("  Im(g_2)       = {:.4f}".format(float(best_params[1])))
print("  Re(epsilon_d) = {:.4f}".format(float(best_params[2])))
print("  Im(epsilon_d) = {:.4f}".format(float(best_params[3])))

In [ ]:
epochs = jnp.arange(N_EPOCHS)
mean_history = jnp.asarray(mean_history)
sample_history = jnp.asarray(sample_history)
sample_reward_history = jnp.asarray(sample_reward_history)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(epochs, epoch_best_reward_history, marker="o")
axes[0, 0].set_title("Best Reward Per Epoch")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Reward")
axes[0, 0].grid(True, alpha=0.3)

labels = [r"Re($g_2$)", r"Im($g_2$)", r"Re($\epsilon_d$)", r"Im($\epsilon_d$)"]
for idx, label in enumerate(labels):
    axes[0, 1].plot(epochs, mean_history[:, idx], marker="o", label=label)
axes[0, 1].set_title("Optimizer Mean Trajectory")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("Control value")
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()

axes[1, 0].semilogy(epochs, epoch_best_tx_history, marker="o", label=r"$T_X$")
axes[1, 0].semilogy(epochs, epoch_best_tz_history, marker="o", label=r"$T_Z$")
axes[1, 0].set_title("Best Estimated Lifetimes")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("Lifetime (us)")
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend()

axes[1, 1].plot(epochs, epoch_best_bias_history, marker="o", label="Estimated bias")
axes[1, 1].axhline(TARGET_BIAS, linestyle="--", color="black", label="Target bias")
axes[1, 1].set_title("Bias Tracking")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel(r"$T_Z / T_X$")
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend()

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sc0 = axes[0].scatter(
    sample_history[:, 0],
    sample_history[:, 1],
    c=sample_reward_history,
    cmap="viridis",
    s=55,
)
axes[0].set_title(r"Sampled $g_2$ Controls")
axes[0].set_xlabel(r"Re($g_2$)")
axes[0].set_ylabel(r"Im($g_2$)")
axes[0].grid(True, alpha=0.3)
fig.colorbar(sc0, ax=axes[0], label="Reward")

sc1 = axes[1].scatter(
    sample_history[:, 2],
    sample_history[:, 3],
    c=sample_reward_history,
    cmap="viridis",
    s=55,
)
axes[1].set_title(r"Sampled $\epsilon_d$ Controls")
axes[1].set_xlabel(r"Re($\epsilon_d$)")
axes[1].set_ylabel(r"Im($\epsilon_d$)")
axes[1].grid(True, alpha=0.3)
fig.colorbar(sc1, ax=axes[1], label="Reward")

plt.tight_layout()
plt.show()

In [ ]:
best_details = evaluate_cat_controls(
    best_params,
    target_bias=TARGET_BIAS,
    kappa_a=KAPPA_A,
    kappa_b=KAPPA_B,
    return_results=True,
)

print("Best solution summary")
print("  reward       = {:.3f}".format(best_details["reward"]))
print("  Tx           = {:.3f} us".format(best_details["Tx"]))
print("  Tz           = {:.3f} us".format(best_details["Tz"]))
print("  bias         = {:.2f}".format(best_details["bias"]))
print("  |alpha|      = {:.3f}".format(best_details["alpha_abs"]))
print("  kappa_2      = {:.3f} MHz".format(best_details["kappa_2"]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(
    best_details["result_z"].tsave,
    best_details["result_z"].expects[1, :].real,
    label=r"Measured $\langle Z_L \rangle$",
)
axes[0].plot(
    best_details["result_z"].tsave,
    best_details["fit_z"]["y_fit"],
    linestyle="--",
    label="Robust exponential fit",
)
axes[0].set_title(r"Logical $Z$ decay at best controls")
axes[0].set_xlabel("Time (us)")
axes[0].set_ylabel("Expectation value")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(
    best_details["result_x"].tsave,
    best_details["result_x"].expects[0, :].real,
    label=r"Measured $\langle X_L \rangle$",
)
axes[1].plot(
    best_details["result_x"].tsave,
    best_details["fit_x"]["y_fit"],
    linestyle="--",
    label="Robust exponential fit",
)
axes[1].set_title(r"Logical $X$ decay at best controls")
axes[1].set_xlabel("Time (us)")
axes[1].set_ylabel("Expectation value")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()